<a href="https://colab.research.google.com/github/nerdycore/Thesis_Project/blob/main/mobilenet_with_mild_augmentation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

# Path to your ZIP file in Drive
zip_path = '/content/drive/MyDrive/data/MangoFruit&Leaf.zip'  # Adjust if needed

# Extract to local VM storage
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("Extraction complete!")

Extraction complete!


In [ ]:

import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import imageio.v2 as io
from skimage.transform import resize
import pathlib
import json



In [ ]:
# data loading
data_dir = pathlib.Path("/content/dataset/MangoFruit&Leaf")
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

mango_disease_dict = {
    "alternaria_fruit": [f for f in data_dir.glob("Alternaria_Fruit/*") if f.suffix.lower() in image_extensions],
    "anthracnose_fruit": [f for f in data_dir.glob("Anthracnose_Fruit/*") if f.suffix.lower() in image_extensions],
    'mango_scab_fruit': [f for f in data_dir.glob("MangoScab_Fruit/*") if f.suffix.lower() in image_extensions],
    "stem_end_rot_fruit": [f for f in data_dir.glob("StemEndrot_Fruit/*") if f.suffix.lower() in image_extensions],
    'healthy_fruit': [f for f in data_dir.glob("Healthy_Fruit/*") if f.suffix.lower() in image_extensions],
    "anthracnose_leaf": [f for f in data_dir.glob("Anthracnose_Leaf/*") if f.suffix.lower() in image_extensions],
    'powdery_mildew_leaf': [f for f in data_dir.glob("PowderyMildew_Leaf/*") if f.suffix.lower() in image_extensions],
    "scooty_mould_leaf": [f for f in data_dir.glob("ScootyMould_Leaf/*") if f.suffix.lower() in image_extensions],
    'healthy_leaf': [f for f in data_dir.glob("Healthy_Leaf/*") if f.suffix.lower() in image_extensions],
}

mango_disease_labels = {
    'alternaria_fruit' : 0, 'anthracnose_fruit': 1, 'mango_scab_fruit': 2,
    "stem_end_rot_fruit": 3, 'healthy_fruit': 4,
    "anthracnose_leaf": 5, 'powdery_mildew_leaf': 6, "scooty_mould_leaf": 7,
    "healthy_leaf": 8
}

In [ ]:
# Load images
x, y = [], []
for disease, images in mango_disease_dict.items():
    for image in images:
        img = io.imread(image)
        resized_img = resize(img, (224, 224), preserve_range=True)
        resized_image = preprocess_input(resized_img.astype(np.float32))
        x.append(resized_image)
        y.append(mango_disease_labels[disease])

x = np.array(x, dtype=np.float32)
y = np.array(y)



In [ ]:
# Split data
x_trainval, x_test, y_trainval, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)
x_train, x_val, y_train, y_val = train_test_split(
    x_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

print(f"Train: {len(x_train)}, Val: {len(x_val)}, Test: {len(x_test)}")

Train: 1728, Val: 432, Test: 540


In [ ]:
# Build MobileNetV2 Model with Mild Data Augmentation

from tensorflow.keras import layers

# Mild augmentation:
# These transformations are intentionally conservative because the previous
# augmentation setup reduced validation/test accuracy substantially.
# Augmentation is applied ONLY during training.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.02),
    layers.RandomZoom(height_factor=0.05, width_factor=0.05),
], name="mild_data_augmentation")

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Keep the same transfer-learning strategy as the original baseline.
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3), name="image_input")

# Random augmentation is active during training and inactive during
# validation/test/prediction.
augmented = data_augmentation(inputs)

x = base_model(augmented, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(9, activation="softmax")(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

with open("mobilenet_mild_augmentation_summary.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mild_data_augmentation          │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 9)              │        11,529 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,269,513 (8.66 MB)

 Trainable params: 11,529 (45.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# Train MobileNetV2 with Mild Data Augmentation

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

y_train_cat = to_categorical(y_train, num_classes=9)
y_val_cat = to_categorical(y_val, num_classes=9)
y_test_cat = to_categorical(y_test, num_classes=9)

checkpoint = ModelCheckpoint(
    "mobilenet_mild_aug_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=15,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

start_time = time.time()

history = model.fit(
    x_train,
    y_train_cat,
    validation_data=(x_val, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks=[checkpoint, early_stopping, reduce_lr],
    verbose=1
)

train_time = time.time() - start_time
print(f"Total training time: {train_time/60:.2f} minutes")


Epoch 1/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3928 - loss: 1.7310
Epoch 1: val_accuracy improved from None to 0.79861, saving model to mobilenet_mild_aug_best.keras

Epoch 1: finished saving model to mobilenet_mild_aug_best.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 17s 129ms/step - accuracy: 0.5793 - loss: 1.1671 - val_accuracy: 0.7986 - val_loss: 0.5683 - learning_rate: 0.0010
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.7862 - loss: 0.5379
Epoch 2: val_accuracy improved from 0.79861 to 0.82176, saving model to mobilenet_mild_aug_best.keras

Epoch 2: finished saving model to mobilenet_mild_aug_best.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.7951 - loss: 0.5182 - val_accuracy: 0.8218 - val_loss: 0.4598 - learning_rate: 0.0010
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.8427 - loss: 0.4304
Epoch 3: val_accuracy did not improve from 0.82176
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.8420 - loss: 0.4082 - va

In [ ]:

# extract metrics

def extract_metrics(history):
    """Extract best epoch and accuracies from history"""
    val_acc = history.history['val_accuracy']
    train_acc = history.history['accuracy']
    best_epoch = np.argmax(val_acc) + 1
    best_val_acc = val_acc[best_epoch - 1]
    best_train_acc = train_acc[best_epoch - 1]

    # Find convergence (where val_acc doesn't improve much)
    # Simple: first epoch where val_acc > 0.80
    converge_epoch = None
    for i, acc in enumerate(val_acc):
        if acc > 0.80:
            converge_epoch = i + 1
            break

    return {
        'best_epoch': best_epoch,
        'best_train_acc': best_train_acc,
        'best_val_acc': best_val_acc,
        'converge_epoch': converge_epoch,
        'total_epochs': len(val_acc)
    }

metrics = extract_metrics(history)
print(f"\n=== MobileNet Results ===")
print(f"Best epoch: {metrics['best_epoch']}")
print(f"Best val accuracy: {metrics['best_val_acc']:.4f}")
print(f"Converged at epoch: {metrics['converge_epoch']}")

# Load best model and evaluate test
model.load_weights('mobilenet_mild_aug_best.keras')
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
metrics['test_acc'] = test_acc
print(f"Test accuracy: {test_acc:.4f}")



=== MobileNet Results ===
Best epoch: 4
Best val accuracy: 0.8333
Converged at epoch: 2
Test accuracy: 0.8426


In [ ]:
# ----------------------------
# 5. Per-class Metrics
# ----------------------------
y_pred = model.predict(x_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
class_report = classification_report(y_test, y_pred_classes, output_dict=True)
metrics['class_report'] = class_report

import json
import numpy as np

def convert_to_serializable(obj):
    """Recursively convert numpy types to Python native types."""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(v) for v in obj]
    else:
        return obj

# Apply conversion before saving
metrics_serializable = convert_to_serializable(metrics)

# Now save
with open('mobilenet_metrics.json', 'w') as f:
    json.dump(metrics_serializable, f, indent=2)
# ----------------------------
# 6. Visualizations
# ----------------------------
# Learning Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].axvline(x=metrics['best_epoch']-1, color='red', linestyle='--', label=f'Best Epoch ({metrics["best_epoch"]})')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Learning Curve - MobileNet')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].axvline(x=metrics['best_epoch']-1, color='red', linestyle='--', label=f'Best Epoch ({metrics["best_epoch"]})')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Curve - MobileNet')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mobilenet_learning_curves.png', dpi=300)
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)
class_names = list(mango_disease_labels.keys())

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - MobileNet')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('mobilenet_confusion_matrix.png', dpi=300)
plt.show()

# ----------------------------
# 7. Save All Results
# ----------------------------
# Summary table
summary = pd.DataFrame({
    'Metric': ['Best Epoch', 'Best Val Accuracy', 'Test Accuracy', 'Total Training Time (min)'],
    'Value': [
        metrics['best_epoch'],
        f"{metrics['best_val_acc']:.4f}",
        f"{metrics['test_acc']:.4f}",
        f"{train_time/60:.2f}"
    ]
})
summary.to_csv('mobilenet_summary.csv', index=False)
print("\nResults saved to 'mobilenet_summary.csv'")

## Experiment notes

This version uses **mild data augmentation** only: horizontal flip, small rotation, and small zoom. The original MobileNetV2 transfer-learning head is restored so the comparison focuses primarily on the effect of augmentation.

**Original baseline:** best validation accuracy = 80.79% and test accuracy = 81.11%.

The previous aggressive augmentation experiment reached only about 71.30% validation accuracy and 70.93% test accuracy, so this version deliberately uses weaker transformations.

After running the notebook, compare the new `Test accuracy` with **81.11%**. The test set is not augmented during evaluation.


In [ ]:
# ----------------------------
# Baseline comparison
# ----------------------------
baseline_test_acc = 0.8111
new_test_acc = metrics.get('test_acc', None)

print(f"Original baseline test accuracy: {baseline_test_acc:.4%}")
if new_test_acc is not None:
    print(f"Mild augmentation test accuracy: {new_test_acc:.4%}")
    print(f"Difference: {(new_test_acc - baseline_test_acc):+.4%}")
    if new_test_acc > baseline_test_acc:
        print("Result: Accuracy improved over the original baseline.")
    elif new_test_acc < baseline_test_acc:
        print("Result: Accuracy decreased; augmentation may still be too strong or not beneficial for this dataset.")
    else:
        print("Result: Accuracy is unchanged from the original baseline.")
